In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
!pip install gensim
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import numpy as np
import nltk
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import torch.nn.functional as F
import torch
import openai

In [ ]:
fn='rt-polarity.neg'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_neg=  content.splitlines()
print ('len of texts_neg = {:,}'.format (len(texts_neg)))
for review in texts_neg[:5]:
    print ( '\n', review)

len of texts_neg = 5,331

 simplistic , silly and tedious . 

 it's so laddish and juvenile , only teenage boys could possibly find it funny . 

 exploitative and largely devoid of the depth or sophistication that would make watching such a graphic treatment of the crimes bearable . 

 [garbus] discards the potential for pathological study , exhuming instead , the skewed melodrama of the circumstantial situation . 

 a visually flashy but narratively opaque and emotionally vapid exercise in style and mystification . 


In [ ]:
fn='rt-polarity.pos'

with open(fn, "r",encoding='utf-8', errors='ignore') as f:
    content = f.read()
texts_pos=  content.splitlines()
print ('len of texts_pos = {:,}'.format (len(texts_pos)))
for review in texts_pos[:5]:
    print ('\n', review)

len of texts_pos = 5,331

 the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal . 

 the gorgeously elaborate continuation of " the lord of the rings " trilogy is so huge that a column of words cannot adequately describe co-writer/director peter jackson's expanded vision of j . r . r . tolkien's middle-earth . 

 effective but too-tepid biopic

 if you sometimes like to go to the movies to have fun , wasabi is a good place to start . 

 emerges as something rare , an issue movie that's so honest and keenly observed that it doesn't feel like one . 


In [ ]:
labels_pos = [1] * len(texts_pos)
labels_neg = [0] * len(texts_neg)

In [ ]:
texts = texts_pos + texts_neg
labels = labels_pos + labels_neg

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size = 0.2, random_state = 42, stratify = labels)

print('Test size: ', len(X_test))
print('Train size: ', len(X_train))

Test size:  2133
Train size:  8529


TF-IDF + Logistic Regression

In [ ]:
tfidf_vectorizer = TfidfVectorizer().fit(X_train)
X_train_vectorized = tfidf_vectorizer.fit_transform(X_train)

In [ ]:
clf = LogisticRegression(max_iter=1000).fit(X_train_vectorized, y_train)
predictions = clf.predict(tfidf_vectorizer.transform(X_test))
scores = clf.decision_function(tfidf_vectorizer.transform(X_test))
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.78      0.76      0.77      1067
           1       0.77      0.78      0.78      1066

    accuracy                           0.77      2133
   macro avg       0.77      0.77      0.77      2133
weighted avg       0.77      0.77      0.77      2133



Word2Vec + Logistic Regression

In [ ]:
nltk.download('punkt_tab')

def tokenize_text(text):
    return word_tokenize(text.lower())

X_train_tokenized = [tokenize_text(text) for text in X_train]

w2v_model = Word2Vec(X_train_tokenized, vector_size=100, window=5, min_count=3, workers=4)
print(f"Vocabulary size: {len(w2v_model.wv.key_to_index)}")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Vocabulary size: 6097


In [ ]:
def document_vector(doc, model):
    doc = [word for word in doc if word in model.wv]
    if len(doc) == 0:
        return np.zeros(model.vector_size)
    return np.mean([model.wv[word] for word in doc], axis=0)

X_train_w2v = np.array([document_vector(doc, w2v_model) for doc in X_train_tokenized])
X_test_tokenized = [tokenize_text(text) for text in X_test]
X_test_w2v = np.array([document_vector(doc, w2v_model) for doc in X_test_tokenized])

In [ ]:
w2v_clf = LogisticRegression(max_iter=1000).fit(X_train_w2v, y_train)
w2v_predictions = w2v_clf.predict(X_test_w2v)
w2v_scores = w2v_clf.predict_proba(X_test_w2v)[:, 1]

print(classification_report(y_test, w2v_predictions))

              precision    recall  f1-score   support

           0       0.58      0.52      0.55      1067
           1       0.56      0.61      0.59      1066

    accuracy                           0.57      2133
   macro avg       0.57      0.57      0.57      2133
weighted avg       0.57      0.57      0.57      2133



Transformer-Based Model

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256
)

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = SentimentDataset(train_encodings, y_train)

test_dataset = SentimentDataset(test_encodings, y_test)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,

    warmup_steps=100,

    weight_decay=0.01,

    logging_dir="./logs",
    logging_steps=10,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    disable_tqdm=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


{'loss': '0.6954', 'grad_norm': '2.904', 'learning_rate': '4.5e-06', 'epoch': '0.01873'}
{'loss': '0.6945', 'grad_norm': '1.569', 'learning_rate': '9.5e-06', 'epoch': '0.03745'}
{'loss': '0.688', 'grad_norm': '1.146', 'learning_rate': '1.45e-05', 'epoch': '0.05618'}
{'loss': '0.6867', 'grad_norm': '1.839', 'learning_rate': '1.95e-05', 'epoch': '0.07491'}
{'loss': '0.6574', 'grad_norm': '4.071', 'learning_rate': '2.45e-05', 'epoch': '0.09363'}
{'loss': '0.6351', 'grad_norm': '5.849', 'learning_rate': '2.95e-05', 'epoch': '0.1124'}
{'loss': '0.6017', 'grad_norm': '2.975', 'learning_rate': '3.45e-05', 'epoch': '0.1311'}
{'loss': '0.4302', 'grad_norm': '7.765', 'learning_rate': '3.95e-05', 'epoch': '0.1498'}
{'loss': '0.4705', 'grad_norm': '9.676', 'learning_rate': '4.45e-05', 'epoch': '0.1685'}
{'loss': '0.4841', 'grad_norm': '8.054', 'learning_rate': '4.95e-05', 'epoch': '0.1873'}
{'loss': '0.4699', 'grad_norm': '9.962', 'learning_rate': '4.97e-05', 'epoch': '0.206'}
{'loss': '0.4274', '

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2381', 'grad_norm': '1.127', 'learning_rate': '3.539e-05', 'epoch': '1.011'}
{'loss': '0.1992', 'grad_norm': '2.82', 'learning_rate': '3.505e-05', 'epoch': '1.03'}
{'loss': '0.2445', 'grad_norm': '16.75', 'learning_rate': '3.472e-05', 'epoch': '1.049'}
{'loss': '0.2059', 'grad_norm': '10.19', 'learning_rate': '3.439e-05', 'epoch': '1.067'}
{'loss': '0.1526', 'grad_norm': '1.201', 'learning_rate': '3.405e-05', 'epoch': '1.086'}
{'loss': '0.1096', 'grad_norm': '12.69', 'learning_rate': '3.372e-05', 'epoch': '1.105'}
{'loss': '0.2934', 'grad_norm': '12.93', 'learning_rate': '3.339e-05', 'epoch': '1.124'}
{'loss': '0.2058', 'grad_norm': '6.508', 'learning_rate': '3.306e-05', 'epoch': '1.142'}
{'loss': '0.2391', 'grad_norm': '6.965', 'learning_rate': '3.272e-05', 'epoch': '1.161'}
{'loss': '0.1975', 'grad_norm': '9.213', 'learning_rate': '3.239e-05', 'epoch': '1.18'}
{'loss': '0.2553', 'grad_norm': '18.69', 'learning_rate': '3.206e-05', 'epoch': '1.199'}
{'loss': '0.2396', 'grad

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.1203', 'grad_norm': '5.427', 'learning_rate': '1.774e-05', 'epoch': '2.004'}
{'loss': '0.07711', 'grad_norm': '0.8716', 'learning_rate': '1.741e-05', 'epoch': '2.022'}
{'loss': '0.08481', 'grad_norm': '3.511', 'learning_rate': '1.708e-05', 'epoch': '2.041'}
{'loss': '0.09817', 'grad_norm': '3.066', 'learning_rate': '1.674e-05', 'epoch': '2.06'}
{'loss': '0.03823', 'grad_norm': '8.833', 'learning_rate': '1.641e-05', 'epoch': '2.079'}
{'loss': '0.04738', 'grad_norm': '1.235', 'learning_rate': '1.608e-05', 'epoch': '2.097'}
{'loss': '0.05482', 'grad_norm': '12.18', 'learning_rate': '1.575e-05', 'epoch': '2.116'}
{'loss': '0.03547', 'grad_norm': '0.07743', 'learning_rate': '1.541e-05', 'epoch': '2.135'}
{'loss': '0.1222', 'grad_norm': '0.8736', 'learning_rate': '1.508e-05', 'epoch': '2.154'}
{'loss': '0.06804', 'grad_norm': '0.0531', 'learning_rate': '1.475e-05', 'epoch': '2.172'}
{'loss': '0.05596', 'grad_norm': '0.1368', 'learning_rate': '1.441e-05', 'epoch': '2.191'}
{'loss'

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


{'train_runtime': '243.4', 'train_samples_per_second': '105.1', 'train_steps_per_second': '6.582', 'train_loss': '0.2495', 'epoch': '3'}


TrainOutput(global_step=1602, training_loss=0.2494709093470159, metrics={'train_runtime': 243.3804, 'train_samples_per_second': 105.132, 'train_steps_per_second': 6.582, 'train_loss': 0.2494709093470159, 'epoch': 3.0})

In [ ]:
predictions = trainer.predict(test_dataset)

preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

           0       0.84      0.86      0.85      1067
           1       0.86      0.84      0.85      1066

    accuracy                           0.85      2133
   macro avg       0.85      0.85      0.85      2133
weighted avg       0.85      0.85      0.85      2133



**Conclusion:** The TF-IDF model achieved an overall accuracy of 77% with balanced precision and recall for both sentiment classes. This approach performed well because TF-IDF effectively captures sentiment-specific keywords that are highly relevant for movie review classification.

The Word2Vec-based approach produced the weakest results with an accuracy of 57%. The lower performance is caused by averaging word embeddings, which loses contextual and sentiment-related information. Logistic Regression also struggles to separate classes effectively using averaged semantic vectors.

The transformer model achieved the best performance with an accuracy of 85% and strong precision/recall balance across both classes. This model performs significantly better but it needs more computational resources and time for training.

The Transformer-Based Model demonstrated the highest effectiveness for sentiment analysis. TF-IDF with Logistic Regression provided strong baseline performance with relatively low computational cost, while Word2Vec with Logistic Regression showed the weakest results due to loss of contextual information during vector averaging.